In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/telco_data.csv")

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['tenure'] = df['tenure'].fillna(df['tenure'].median())
df['MonthlyCharges'] = df['MonthlyCharges'].fillna(df['MonthlyCharges'].median())
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
for col in ['gender', 'Partner', 'InternetService', 'StreamingTV']:
    df[col] = df[col].fillna(df[col].mode()[0])

df.isnull().sum().sum()  # should print 0

np.int64(0)

In [4]:
df['charge_ratio'] = df['MonthlyCharges'] / (df['TotalCharges'] / df['tenure'].replace(0, 1))
df['tenure_group'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72], labels=['0-1yr', '1-2yr', '2-4yr', '4-6yr'])

df[['tenure', 'MonthlyCharges', 'TotalCharges', 'charge_ratio', 'tenure_group']].head()

,tenure,MonthlyCharges,TotalCharges,charge_ratio,tenure_group
0,1.0,29.85,29.85,1.000000,0-1yr
1,34.0,56.95,1889.50,1.024768,2-4yr
2,2.0,53.85,108.15,0.995839,0-1yr
3,45.0,42.30,1840.75,1.034089,2-4yr
4,2.0,70.70,151.65,0.932410,0-1yr


In [5]:
# Total number of add-on services a customer has (more services = often more "sticky")
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['total_services'] = (df[service_cols] == 'Yes').sum(axis=1)

# Difference between what they've paid total vs. what tenure*monthly would predict
df['charges_difference'] = df['TotalCharges'] - (df['tenure'] * df['MonthlyCharges'])

df[['total_services', 'charges_difference']].head()

,total_services,charges_difference
0,1,0.00
1,2,-46.80
2,2,0.45
3,3,-62.75
4,0,10.25


In [8]:
from sklearn.preprocessing import LabelEncoder

# Convert target variable: Yes/No -> 1/0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Identify all remaining text (categorical) columns except customerID
categorical_cols = df.select_dtypes(include='object').columns.tolist()
categorical_cols.remove('customerID')

# Label encode each one
le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col].astype(str))

df.head()

C:\Users\zaidm\AppData\Local\Temp\ipykernel_10336\1310214721.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include='object').columns.tolist()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,charge_ratio,tenure_group,total_services,charges_difference
0,7590-VHVEG,0,0,1,0,1.0,0,1,0,0,...,0,1,2,29.85,29.85,NaN,1.000000,0-1yr,1,0.00
1,5575-GNVDE,1,0,0,0,34.0,1,0,0,2,...,1,0,3,56.95,1889.50,NaN,1.024768,2-4yr,2,-46.80
2,3668-QPYBK,1,0,0,0,2.0,1,0,0,2,...,0,1,3,53.85,108.15,NaN,0.995839,0-1yr,2,0.45
3,7795-CFOCW,1,0,0,0,45.0,0,1,0,2,...,1,0,0,42.30,1840.75,NaN,1.034089,2-4yr,3,-62.75
4,9237-HQITU,0,0,0,0,2.0,1,0,1,0,...,0,1,2,70.70,151.65,NaN,0.932410,0-1yr,0,10.25


In [9]:
df.to_csv("../data/processed/telco_processed.csv", index=False)
print("Saved processed dataset:", df.shape)

OSError: Cannot save file into a non-existent directory: '..\data\processed'

In [10]:
df['Churn'].unique()

array([nan])

In [11]:
import os
os.makedirs("../data/processed", exist_ok=True)

df.to_csv("../data/processed/telco_processed.csv", index=False)
print("Saved processed dataset:", df.shape)

Saved processed dataset: (7043, 25)
